In [1]:
import os
os.environ["MKL_NUM_THREADS"] = "5"
os.environ["OPENBLAS_NUM_THREADS"] = "5"
os.environ["NUMEXPR_NUM_THREADS"] = "5"

# Example GLM behaviour

In [3]:
from importlib.resources import files
import pandas as pd
import numpy as np
from lepto.behaviour.model.linear_demand import GLMDemand
from lepto.standard.model.linear_model import transform_json_to_df

## Data import

In [4]:
path = files("lepto.data") / "sample_behaviour.csv"
df = pd.read_csv(path)
        

## Model

In [7]:
X = df[["age", "income", "region", "promo", "loyalty", "price"]]
w = df['w']
y = df['y']

penalty_static = {'age': {'penalty':'continuous'}, 'income': {'penalty':'continuous'}, 'region': {'penalty':'categorical'}}
penalty_elasticity = {'promo': {'penalty':'categorical'},
           'loyalty': {'penalty':'continuous'}}

glm = GLMDemand(
    var_glm_static=["age", "income", "region"],
    var_glm_elasticity=["promo", "loyalty"],
    var_behaviour="price",
    direction="increasing",
    static_penalty_choice=penalty_static,
    elasticity_penalty_choice=penalty_elasticity,
    nbins=10,
    lam=1e2,
    lam_behaviour=1e5,
    sparse_output=True
)

glm.fit(X, y, sample_weight=w)
pred = glm.predict(X)
glm.summary

{'static': {'intercept': np.float64(-2.826642164673915),
  'coefficients': {'age': {'(24, 30]': np.float64(0.020594345395247713),
    '(30, 36]': np.float64(0.04525141748705722),
    '(36, 42]': np.float64(0.08010754718498696),
    '(42, 48]': np.float64(0.11186301820882336),
    '(48, 54]': np.float64(0.15583592937219015),
    '(54, 61]': np.float64(0.22290883416374116),
    '(61, 67]': np.float64(0.2790803191046541),
    '(67, 73]': np.float64(0.30595946533347773),
    '(73, 79]': np.float64(0.33565227055133684)},
   'income': {'(20551.5, 27518.6]': np.float64(0.0010898736212000952),
    '(27518.6, 32083.3]': np.float64(0.03853509212531728),
    '(32083.3, 36119.1]': np.float64(0.10407818329703965),
    '(36119.1, 39695.7]': np.float64(0.18931833549645125),
    '(39695.7, 43385.5]': np.float64(0.2754103504379093),
    '(43385.5, 47776.9]': np.float64(0.3286479103913793),
    '(47776.9, 52585.2]': np.float64(0.3716730632224553),
    '(52585.2, 59441.7]': np.float64(0.43910498703476836

In [6]:
glm.plot(X, y, w, "age", "static")

In [7]:
glm.plot(X, y, w, "loyalty", "elasticity")

In [8]:
# Model to excel format
transform_json_to_df(glm.summary['static'])

,col0,col1,col2,col3,col4,col5,col6,col7,col8
0,Base,,-2.749013,,,,,,
1,,,,,,,,,
2,,,,,,,,,
3,age,,,income,,,region,,
4,,,,,,,,,
5,age,,,income,,,region,,
6,"(24, 30]",0.001851,,"(20551.5, 27518.6]",-0.021857,,1.0,0.00813,
7,"(30, 36]",0.018282,,"(27518.6, 32083.3]",0.001821,,2.0,-0.016289,
8,"(36, 42]",0.04698,,"(32083.3, 36119.1]",0.053318,,3.0,-0.005964,
9,"(42, 48]",0.070581,,"(36119.1, 39695.7]",0.128783,,,,


In [9]:
glm.model.coef_glm1_

array([ 1.85138852e-03,  1.82820892e-02,  4.69796149e-02,  7.05805850e-02,
        1.07544354e-01,  1.67891890e-01,  2.19627242e-01,  2.50560677e-01,
        2.84790117e-01, -2.18571608e-02,  1.82123043e-03,  5.33179378e-02,
        1.28782923e-01,  2.12318642e-01,  2.64420334e-01,  3.07897528e-01,
        3.79290238e-01,  4.04955985e-01,  8.13015490e-03, -1.62889648e-02,
       -5.96439875e-03, -2.74901300e+00])

## Offset model

In [ ]:
# Offset model to force one coefficient to be at specific values. You need to know the position of the coefficient in the model. It can be done via glm.model.coef_glm1_
offset_static = np.array([np.nan, np.nan,  np.nan,  np.nan,  np.nan,
        np.nan,  np.nan, np.nan,  np.nan, np.nan,
       np.nan, np.nan, np.nan ,  np.nan,  np.nan,
        np.nan,  np.nan , np.nan,  1.5, np.nan,
        np.nan,  np.nan])

glm = GLMDemand(
    var_glm_static=["age", "income", "region"],
    var_glm_elasticity=["promo", "loyalty"],
    var_behaviour="price",
    direction="increasing",
    static_penalty_choice=penalty_static,
    elasticity_penalty_choice=penalty_elasticity,
    nbins=10,
    lam=1e2,
    lam_behaviour=1e5,
    static_offset_betas=offset_static,
    elasticity_offset_betas=None,
)

glm.fit(X, y, sample_weight=w)
pred = glm.predict(X)
glm.summary

{'static': {'intercept': np.float64(-3.5753854738768367),
  'coefficients': {'age': {'(24, 30]': np.float64(0.017198289499222502),
    '(30, 36]': np.float64(0.01958681940671274),
    '(36, 42]': np.float64(0.029664733069055618),
    '(42, 48]': np.float64(0.05139516561696923),
    '(48, 54]': np.float64(0.09041768171083718),
    '(54, 61]': np.float64(0.15450771288066723),
    '(61, 67]': np.float64(0.22314833631903497),
    '(67, 73]': np.float64(0.26008756591849713),
    '(73, 79]': np.float64(0.30656593111680724)},
   'income': {'(20551.5, 27518.6]': np.float64(0.00484518691850648),
    '(27518.6, 32083.3]': np.float64(0.03202379521084103),
    '(32083.3, 36119.1]': np.float64(0.08059122692539698),
    '(36119.1, 39695.7]': np.float64(0.15508602293860846),
    '(39695.7, 43385.5]': np.float64(0.23821625770480814),
    '(43385.5, 47776.9]': np.float64(0.28991294649838983),
    '(47776.9, 52585.2]': np.float64(0.3311123979429291),
    '(52585.2, 59441.7]': np.float64(0.40113003660973